# MERIT — Layer 3: utility-ranked retrieval

**Result: negative, with a precise diagnosis.** Ranking retrieval candidates by
global per-entry utility does *not* beat plain relevance on this fixture. The
utility estimates are correct — distractors are learned as ~0 and dropped — but
global utility carries no information about *which query* an entry helps.

This notebook reproduces that result and the evidence for the cause. Mock only,
zero tokens.

In [ ]:
import sys, pathlib, contextlib, io
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()))
import pandas as pd
pd.set_option("display.width", 180)
from merit import Config, UtilityPolicy, Layer1Runner, build_store, build_tasks
from merit.clients import MockClient
from merit import evaluate as E

store, tasks = build_store(), build_tasks()
SEED, N = 20261006, 480
def quiet(f):
    with contextlib.redirect_stdout(io.StringIO()):
        return f()

## 1. Where relevance gets it wrong

Tasks where BM25's top-ranked entry is *not* the one carrying the needed flag.
With a single memory slot, relevance fails every one of these by construction.

In [ ]:
misranked = E.misranked_tasks(store, tasks)
print(len(misranked), "of", len(tasks), "tasks:", sorted(misranked))

## 2. Three policies, two settings

- **relevance** — top-k by BM25, the baseline
- **thompson** — sample utility from the posterior, x relevance / tokens
- **greedy** — posterior mean x relevance; unfitted entries neutral; proven dead weight dropped

E1 uses one memory slot; E2 uses four. Masking stays on at 10% in every arm.

In [ ]:
def mk(run, k):
    return Config(run_id=run, backend="mock", mask_p=0.10, top_k=k,
                  max_turns=N, seed=SEED, db_path="data/layer3.db")

def three_arms(k):
    arms = [
        quiet(lambda: E.run_arm("relevance", mk(f"rel_k{k}", k), None, MockClient(0.9, SEED)))[0],
        quiet(lambda: E.run_arm("thompson", mk(f"ts_k{k}", k),
              lambda s: UtilityPolicy(s, mode="thompson", seed=SEED), MockClient(0.9, SEED)))[0],
        quiet(lambda: E.run_arm("greedy", mk(f"gr_k{k}", k),
              lambda s: UtilityPolicy(s, mode="greedy", prior_mean=0.2, seed=SEED), MockClient(0.9, SEED)))[0],
    ]
    return pd.concat(arms)

e1 = three_arms(1)
E.summarize(e1, misranked)

In [ ]:
E.learning_curve(e1, misranked, window=120)

Greedy *learns* on the misranked tasks — 0.00 rising to ~0.37 — but loses
ground on the tasks relevance already got right. Thompson never settles: entries
without enough withheld trials stay on a wide prior and keep winning draws.

In [ ]:
e2 = three_arms(4)
E.summarize(e2, misranked)

## 3. Diagnosis: what does greedy inject on the tasks it now gets wrong?

In [ ]:
cfg = mk("diag", 1)
pol = UtilityPolicy(store, mode="greedy", prior_mean=0.2, seed=SEED)
def go():
    with Layer1Runner(cfg, store=store, tasks=tasks, client=MockClient(0.9, SEED), policy=pol) as r:
        r.run(verbose=False)
        return r.ledger.dataframe("diag")
L = quiet(go)

need = {t.task_id: t.required_artifact for t in tasks}
art = {e: v.artifact for e, v in store.entries.items()}
pl = {e: v.planted for e, v in store.entries.items()}

inj = L[L.masked == 0]
late = inj[inj.index >= inj.index.max() * 0.5]
late = late[~late.task_id.isin(misranked)]
def kind(r):
    if art[r.entry_id] == need[r.task_id]: return "correct entry"
    if pl[r.entry_id] == "useful":        return "useful entry, WRONG task family"
    return pl[r.entry_id]
late.apply(kind, axis=1).value_counts(normalize=True).round(3)

In [ ]:
{k: round(v.mean, 2) for k, v in pol.post.items()}

## 4. What this means

**The utility signal is correct.** Distractors are learned as ~0 and dropped;
not one appears in the late injections. Layer 2 works.

**Global utility is the wrong object for per-query retrieval.** An entry's
utility is averaged over every turn it was a candidate on. `e_build_flag` is
valuable *on build tasks*; ranked by global utility it gets pushed onto test
questions, where it does nothing. Relevance, multiplied in, is not a strong
enough signal of context to stop that.

**Where global utility *is* the right object: lifecycle.** "Is this entry worth
keeping in the store at all, or promoting into the always-injected core?" is a
question about the entry, not about a query. That is Layer 4, and Layer 2
already showed utility beats recall frequency there.

**What a working Layer 3 needs:** utility conditional on the query — a
contextual bandit — and a support constraint, so an entry is never pushed into
a context where it was never measured. That design should be chosen on real
data, where the right query features are not guessable from a fixture.